# 04 escala e cobertura

Checa (1) cobertura desigual entre colunas e (2) diferença de grandezas.

Saída: reports/escala_cobertura.txt

In [1]:
# Configuração (caminhos e constantes do projeto)
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV = ROOT / "data set.csv"
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
RAW_PARQUET = DATA_DIR / "raw.parquet"

SEED = 42
TARGET = "CLASSI_FIN"
DATE_COLS = ["DT_NOTIFIC", "DT_SIN_PRI", "DT_NASC", "DT_INVEST", "DT_ENCERRA"]

In [2]:
import numpy as np, pandas as pd
out = open(REPORTS_DIR / "escala_cobertura.txt", "w")
def log(*a):
    print(*a); print(*a, file=out)
pd.set_option("display.width", 200, "display.max_rows", 200)

raw = pd.read_parquet(RAW_PARQUET)
raw = raw[raw.NU_ANO.between(2013, 2020)]
log("=== 1. COBERTURA (bruto, 2013-2020) ===")
log("Linhas:", len(raw))
IGN = {"CS_SEXO": 9, "CS_GESTANT": 9, "CS_RACA": 9, "CS_ESCOL_N": [9, 10], "TPAUTOCTO": 9, "HOSPITALIZ": 9,
       "EVOLUCAO": 9, "FEBRE": 9, "RESUL_SORO": 4, "RESUL_NS1": 4, "RESUL_VI_N": 4, "RESUL_PCR_": 4,
       "HISTOPA_N": 4, "IMUNOH_N": 4}
rows = []
for c in raw.columns:
    r = {"col": c, "NaN_%": 100 * raw[c].isna().mean()}
    if c in IGN:
        r["ignorado/nao_realizado_%"] = 100 * raw[c].isin(np.atleast_1d(IGN[c])).mean()
    rows.append(r)
cov = pd.DataFrame(rows).fillna(0).round(2)
cov["info_util_%"] = (100 - cov["NaN_%"] - cov["ignorado/nao_realizado_%"]).round(2)
log(cov.sort_values("info_util_%").to_string(index=False))

=== 1. COBERTURA (bruto, 2013-2020) ===
Linhas: 6732103


       col  NaN_%  ignorado/nao_realizado_%  info_util_%
 HISTOPA_N   0.00                     99.95         0.05
RESUL_VI_N   0.00                     99.94         0.06
  IMUNOH_N   0.00                     99.93         0.07
RESUL_PCR_   0.00                     99.90         0.10
RESUL_SORO   0.00                     98.64         1.36
 RESUL_NS1   0.00                     97.60         2.40
CS_ESCOL_N   0.00                     64.49        35.51
     FEBRE   0.00                     52.41        47.59
 TPAUTOCTO   0.00                     44.16        55.84
HOSPITALIZ   0.00                     43.52        56.48
   CS_RACA   0.00                     34.27        65.73
CS_GESTANT   0.00                     10.71        89.29
  EVOLUCAO   0.00                      9.28        90.72
   DT_NASC   5.79                      0.00        94.21
 DT_INVEST   0.02                      0.00        99.98
DT_ENCERRA   0.01                      0.00        99.99
   COUFINF   0.00              

## 2. GRANDEZAS (bruto, colunas numéricas)

In [3]:
log("\n=== 2. GRANDEZAS (bruto, colunas numéricas) ===")
num = raw.select_dtypes("number")
st = num.agg(["min", "max", "mean", "std"]).T
st["amplitude"] = st["max"] - st["min"]
log(st.sort_values("amplitude", ascending=False).round(2).to_string())
log("\nRazão maior/menor desvio-padrão (bruto):", round(st["std"].max() / st[st["std"] > 0]["std"].min(), 1))


=== 2. GRANDEZAS (bruto, colunas numéricas) ===


                min        max        mean         std  amplitude
ID_UNIDADE      9.0  9999999.0  3207249.14  2406769.70  9999990.0
ID_MUNICIP      9.0   613592.0   337844.36    90912.10   613583.0
COMUNINF        9.0   539934.0   181096.63   187473.51   539925.0
ID_MN_RESI  35050.0   539927.0   337835.95    90907.72   504877.0
SEM_PRI      1352.0   202053.0   201645.89     1119.15   200701.0
SEM_NOT      1401.0   202053.0   201653.60     1011.91   200652.0
ID_RG_RESI      9.0     6262.0     1437.13      706.10     6253.0
ID_REGIONA      9.0     6256.0     1436.21      704.50     6247.0
NU_IDADE_N      0.0      999.0       32.41       18.40      999.0
COPAISINF       0.0      175.0        4.86        4.01      175.0
SG_UF           9.0       53.0       33.61        9.09       44.0
COUFINF         9.0       53.0       22.43       14.54       44.0
SG_UF_NOT      11.0       53.0       33.61        9.09       42.0
CS_ESCOL_N      0.0       10.0        7.49        2.51       10.0
CS_SEXO   

In [4]:
for cen in ["binario", "multiclasse"]:
    log(f"\n=== 3. CENÁRIO {cen}: desvio-padrão e amplitude por variante (treino) ===")
    d = DATA_DIR / "processed" / cen
    if not (d / "train_tree.parquet").exists():
        log(f"(cenário {cen} não gerado — rode o notebook 02 com CENARIO={cen!r} para incluí-lo)"); continue
    res = {}
    for k in ["tree", "std", "minmax"]:
        t = pd.read_parquet(d / f"train_{k}.parquet").drop(columns="y")
        res[k] = pd.DataFrame({"min": t.min(), "max": t.max(), "std": t.std()})
    t = res["tree"]
    log("[tree - sem escala] colunas com amplitude > 1:")
    log(t[(t["max"] - t["min"]) > 1].round(3).to_string())
    log("razão max(std)/min(std) entre colunas não constantes:")
    for k, r in res.items():
        rr = r[r["std"] > 0]["std"]
        log(f"  {k:7s}: {rr.max()/rr.min():8.1f}   (maior std: {rr.idxmax()}={rr.max():.3f}; menor: {rr.idxmin()}={rr.min():.4f})")
    s = res["std"].loc[["idade", "dias_sintoma_notif"]]
    log("[std] contínuas:"); log(s.round(3).to_string())
    m = res["minmax"].loc[["idade", "dias_sintoma_notif"]]
    log("[minmax] contínuas:"); log(m.round(3).to_string())
out.close()


=== 3. CENÁRIO binario: desvio-padrão e amplitude por variante (treino) ===


[tree - sem escala] colunas com amplitude > 1:
                    min   max     std
idade               0.0  99.0  18.511
dias_sintoma_notif  0.0  30.0   3.924
mes_sin            -1.0   1.0   0.547
mes_cos            -1.0   1.0   0.628
razão max(std)/min(std) entre colunas não constantes:
  tree   :    528.1   (maior std: idade=18.511; menor: sexo_ign=0.0351)
  std    :     28.5   (maior std: idade=1.000; menor: sexo_ign=0.0351)
  minmax :     17.9   (maior std: mes_cos=0.628; menor: sexo_ign=0.0351)
[std] contínuas:
                      min    max  std
idade              -1.753  3.596  1.0
dias_sintoma_notif -0.863  6.783  1.0
[minmax] contínuas:
                    min  max    std
idade               0.0  1.0  0.187
dias_sintoma_notif  0.0  1.0  0.131

=== 3. CENÁRIO multiclasse: desvio-padrão e amplitude por variante (treino) ===


[tree - sem escala] colunas com amplitude > 1:
                    min   max     std
idade               0.0  99.0  18.382
dias_sintoma_notif  0.0  30.0   4.007
mes_sin            -1.0   1.0   0.538
mes_cos            -1.0   1.0   0.622
razão max(std)/min(std) entre colunas não constantes:
  tree   :    484.6   (maior std: idade=18.382; menor: sexo_ign=0.0379)
  std    :     26.4   (maior std: dias_sintoma_notif=1.000; menor: sexo_ign=0.0379)
  minmax :     16.4   (maior std: mes_cos=0.622; menor: sexo_ign=0.0379)
[std] contínuas:
                      min    max  std
idade              -1.763  3.623  1.0
dias_sintoma_notif -0.848  6.638  1.0
[minmax] contínuas:
                    min  max    std
idade               0.0  1.0  0.186
dias_sintoma_notif  0.0  1.0  0.134
